# MAVE-Q: Modular Autonomous Verified Evolution — Kaggle (`T4 x2`) & Google Colab (`T4`) Training Notebook

> **Paper**: *MAVE-Q: Modular Autonomous Verified Evolution for Small-Model Quantitative Research in Rule-Constrained Emerging Markets*  
> **Author**: **Tran Duy Khang** — *AI Software Engineer / Freelance Vibe Researcher* ([@KenyaWashed](https://github.com/KenyaWashed))  
> **Disclaimer**: Academic research & simulation ONLY. Zero live trading, zero brokerage APIs, zero investment advice.

### What This Notebook Trains (100% FREE Cloud GPU — $0)
| Model | Architecture | Calibrated Size (`kaggle_t4_optimal`) | Est. Training VRAM | Device (Kaggle `T4 x2` / Colab `T4`) |
| :--- | :--- | :---: | :---: | :---: |
| **Model 1: Narrator / IdeaMiner** | `Qwen3-4B-bnb-4bit` *(or `Qwen3-1.7B-bnb-4bit`)* via Unsloth 4-bit QLoRA | `4.0B` *(or `1.7B`)* | `~4.2–4.5 GB` *(`~2.0 GB`)* | `cuda:0` / `cuda:0` |
| **Model 2: Factor Synthesizer & Mutator** | **Discrete Masked AST Diffusion (Tree-MDLM)** w/ Subtree Infilling + ReST | **6,358,082 params (`6.36M`)** | **~129 MB** | `cuda:1` / `cuda:0` |
| **Model 3: Regime Planner & Verifier** | **HRM / TRM Hierarchical Recursive Reasoner** (Slow $z_H$ + Fast $z_L$) | **3,377,927 params (`3.38M`)** | **~76 MB** | `cuda:1` / `cuda:0` |

In [ ]:
# Cell 1: Inspect Free GPU Hardware (Kaggle T4 x2 = 2x 16GB | Colab T4 = 1x 16GB)
import os, torch
print('Platform:', 'Kaggle' if os.path.exists('/kaggle/working') else ('Colab' if os.path.exists('/content') else 'Local'))
print('CUDA available:', torch.cuda.is_available(), '| GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} ({p.total_memory / 1024**3:.2f} GB VRAM)')
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv

In [ ]:
# Cell 2: Clone MAVE-Q Repository from GitHub (update REPO_URL if you renamed your repo)
import os
REPO_URL = 'https://github.com/KenyaWashed/VN_Trading_Agents.git'
REPO_DIR = 'VN_Trading_Agents'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -n 1 --oneline

In [ ]:
# Cell 3: Install Core Package + Unsloth (4-bit QLoRA on Free T4)
!pip install -q -e '.[dev]'
!pip install -q unsloth trl datasets bitsandbytes accelerate
# Optional: uncomment next line ONLY if you want to fetch your own empirical vnstock snapshot
# !pip install -q vnstock

In [ ]:
# Cell 4: Run Mandatory Anti-Leakage, Causality, and Sandbox Seal Tests BEFORE Training
!pytest tests/ -q

In [ ]:
# Cell 5: Configure Training Hyperparameters (Edit Here for Experiments)
PRESET = 'kaggle_t4_optimal'            # 'kaggle_t4_optimal' (~9.74M micro-core) | 'cpu_fast'
SLM_MODEL = 'unsloth/Qwen3-4B-bnb-4bit' # Use 'unsloth/Qwen3-1.7B-bnb-4bit' for ~2x faster Colab T4 runs
ENABLE_UNSLOTH_QLORA = True             # Fine-tune Model 1 (SLM Narrator) on verified search-zone trajectories
MICRO_EPOCHS = 25                       # ReST epochs for Discrete AST Diffusion (6.36M) + HRM Reasoner (3.38M)
OUTER_ITERS = 4                         # Bi-level outer RSI iterations
SEED = 2026
CKPT_DIR = '/kaggle/working/mave_q_checkpoints' if os.path.exists('/kaggle/working') else 'artifacts/checkpoints'
print('Checkpoints will be saved to:', CKPT_DIR)

In [ ]:
# Cell 6: Phase 1-3 Snapshot + Baselines + Agent v0 Setup
!python experiments/run_data.py
!python experiments/run_baselines.py --reset-ledger
!python experiments/run_agent_v0.py

In [ ]:
# Cell 7: Train Modular Micro-RSI Core (Discrete AST Diffusion 6.36M + HRM 3.38M + Qwen3-4B 4-bit QLoRA)
qlora_flag = '--enable-unsloth-qlora' if ENABLE_UNSLOTH_QLORA else ''
!python train_colab_kaggle.py \
    --preset {PRESET} \
    --slm-model {SLM_MODEL} \
    {qlora_flag} \
    --epochs {MICRO_EPOCHS} \
    --outer-iters {OUTER_ITERS} \
    --seed {SEED} \
    --checkpoint-dir {CKPT_DIR}

In [ ]:
# Cell 8: Run Monolithic RSI Ablations + Modular Micro-RSI Sealed Evaluation + Auto-Verify Paper Tables
!python experiments/run_rsi.py
!python experiments/run_modular_rsi.py --preset {PRESET}
!python experiments/build_report.py
!python paper/verify_paper.py

In [ ]:
# Cell 9: Inspect Auto-Generated Paper Comparison Table & Saved Checkpoints
from pathlib import Path
print('=== paper/tables/main_results.tex ===')
print(Path('paper/tables/main_results.tex').read_text(encoding='utf-8'))
print('=== paper/tables/rsi_ablations.tex ===')
print(Path('paper/tables/rsi_ablations.tex').read_text(encoding='utf-8'))
print('=== Saved Checkpoints ===')
for p in sorted(Path(CKPT_DIR).rglob('*')):
    if p.is_file():
        print(f'  {p} ({p.stat().st_size / 1024**2:.2f} MB)')